In [3]:
import os
import platform
import subprocess
import multiprocessing as mp
from pathlib import Path

import torch


def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT)
        return out.strip()
    except Exception as e:
        return f"ERROR: {e}"


def get_ram_gb():
    try:
        import psutil
        mem = psutil.virtual_memory()
        return mem.total / 1024**3, mem.available / 1024**3
    except Exception:
        return None, None


def recommend_jobs(
    gpu_mem_total_gb,
    gpu_mem_free_gb,
    cpu_count,
    batch_size=256,
    model_size="mixed",
):
    """
    Rough safe heuristic for your Transformer grid.

    model_size:
        small  — mostly T_small_mean
        base   — mostly T_base/T_base_cls
        mixed  — full grid including T_wide_mean/T_deep_mean
        wide   — mostly T_wide_mean
    """

    if gpu_mem_total_gb is None:
        return {
            "recommended_jobs": 1,
            "reason": "CUDA/GPU not detected. Use CPU mode.",
            "batch_size": min(batch_size, 64),
            "num_workers": 0,
        }

    # Conservative per-process memory estimate for your models with AMP.
    # These are intentionally cautious.
    if model_size == "small":
        mem_per_job = 4.0 if batch_size <= 256 else 6.0
    elif model_size == "base":
        mem_per_job = 6.0 if batch_size <= 256 else 8.0
    elif model_size == "wide":
        mem_per_job = 9.0 if batch_size <= 256 else 12.0
    else:
        mem_per_job = 8.0 if batch_size <= 256 else 10.0

    # Leave safety reserve for CUDA context, dataloaders, fragmentation.
    reserve_gb = max(6.0, 0.15 * gpu_mem_total_gb)
    usable_gb = max(0.0, gpu_mem_free_gb - reserve_gb)

    by_mem = int(usable_gb // mem_per_job)
    by_cpu = max(1, cpu_count // 4)

    recommended = max(1, min(by_mem, by_cpu))

    # Clamp to sensible range for one A100.
    if gpu_mem_total_gb >= 70:
        recommended = min(recommended, 6)
    elif gpu_mem_total_gb >= 35:
        recommended = min(recommended, 4)
    else:
        recommended = min(recommended, 2)

    if recommended <= 1:
        safe_batch = min(batch_size, 512)
        num_workers = 2
    elif recommended <= 2:
        safe_batch = min(batch_size, 256)
        num_workers = 2
    elif recommended <= 4:
        safe_batch = min(batch_size, 192)
        num_workers = 1
    else:
        safe_batch = min(batch_size, 128)
        num_workers = 1

    reason = (
        f"Estimated usable GPU memory: {usable_gb:.1f} GB after reserve; "
        f"estimated memory per job: {mem_per_job:.1f} GB; "
        f"CPU limit suggests up to {by_cpu} jobs."
    )

    return {
        "recommended_jobs": recommended,
        "reason": reason,
        "batch_size": safe_batch,
        "num_workers": num_workers,
        "usable_gpu_mem_gb": usable_gb,
        "estimated_mem_per_job_gb": mem_per_job,
        "cpu_limited_jobs": by_cpu,
        "memory_limited_jobs": by_mem,
    }


def inspect_system(batch_size=256, model_size="mixed"):
    print("=" * 90)
    print("SYSTEM")
    print("=" * 90)
    print("Platform:", platform.platform())
    print("Python:", platform.python_version())
    print("Working dir:", Path.cwd())
    print("CPU cores:", os.cpu_count())
    print("CPU count via multiprocessing:", mp.cpu_count())

    ram_total, ram_available = get_ram_gb()
    if ram_total is not None:
        print(f"RAM total: {ram_total:.2f} GB")
        print(f"RAM available: {ram_available:.2f} GB")
    else:
        print("RAM: psutil not available")

    print("\n" + "=" * 90)
    print("PYTORCH / CUDA")
    print("=" * 90)
    print("torch version:", torch.__version__)
    print("torch CUDA version:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())

    gpu_mem_total_gb = None
    gpu_mem_free_gb = None

    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("CUDA device count:", n)

        for i in range(n):
            props = torch.cuda.get_device_properties(i)
            total_gb = props.total_memory / 1024**3

            torch.cuda.set_device(i)
            free_bytes, total_bytes = torch.cuda.mem_get_info(i)
            free_gb = free_bytes / 1024**3
            total_runtime_gb = total_bytes / 1024**3

            print("\nGPU", i)
            print("Name:", props.name)
            print(f"Total memory by properties: {total_gb:.2f} GB")
            print(f"Free memory now: {free_gb:.2f} GB")
            print(f"Total memory runtime: {total_runtime_gb:.2f} GB")
            print("Compute capability:", f"{props.major}.{props.minor}")
            print("Multiprocessors:", props.multi_processor_count)

            if i == 0:
                gpu_mem_total_gb = total_runtime_gb
                gpu_mem_free_gb = free_gb
    else:
        print("No CUDA GPU visible to PyTorch.")

    print("\n" + "=" * 90)
    print("NVIDIA-SMI")
    print("=" * 90)
    print(run_cmd("nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu --format=csv,noheader,nounits"))

    print("\n" + "=" * 90)
    print("RECOMMENDATION")
    print("=" * 90)

    rec = recommend_jobs(
        gpu_mem_total_gb=gpu_mem_total_gb,
        gpu_mem_free_gb=gpu_mem_free_gb,
        cpu_count=os.cpu_count() or 1,
        batch_size=batch_size,
        model_size=model_size,
    )

    print("Model size assumption:", model_size)
    print("Input batch_size:", batch_size)
    print("Recommended MAX_PARALLEL_JOBS:", rec["recommended_jobs"])
    print("Recommended BATCH_SIZE:", rec["batch_size"])
    print("Recommended NUM_WORKERS:", rec["num_workers"])
    print("Reason:", rec["reason"])

    print("\nSuggested config:")
    print(f"MAX_PARALLEL_JOBS = {rec['recommended_jobs']}")
    print(f"BATCH_SIZE = {rec['batch_size']}")
    print(f"NUM_WORKERS = {rec['num_workers']}")
    print("USE_AMP = True")
    print("GPU_IDS = [0]")

    if torch.cuda.is_available() and gpu_mem_total_gb is not None:
        if gpu_mem_total_gb >= 70:
            print("\nA100 80GB-like GPU detected: you can test 4–6 jobs.")
        elif gpu_mem_total_gb >= 35:
            print("\nA100 40GB-like GPU detected: start with 2–3 jobs, maybe 4 if memory/utilization allow.")
        else:
            print("\nSmaller GPU detected: use 1–2 jobs.")

    return rec


# ============================================================
# RUN
# ============================================================

rec = inspect_system(
    batch_size=256,
    model_size="mixed",  # small / base / mixed / wide
)

SYSTEM
Platform: Linux-6.8.0-110-generic-x86_64-with-glibc2.39
Python: 3.12.3
Working dir: /home/user/fractional_unlearning
CPU cores: 32
CPU count via multiprocessing: 32
RAM total: 30.79 GB
RAM available: 23.61 GB

PYTORCH / CUDA
torch version: 2.6.0+cu124
torch CUDA version: 12.4
CUDA available: True
CUDA device count: 1

GPU 0
Name: NVIDIA A100-PCIE-40GB
Total memory by properties: 39.38 GB
Free memory now: 38.95 GB
Total memory runtime: 39.38 GB
Compute capability: 8.0
Multiprocessors: 108

NVIDIA-SMI
NVIDIA A100-PCIE-40GB, 40960, 437, 39890, 0

RECOMMENDATION
Model size assumption: mixed
Input batch_size: 256
Recommended MAX_PARALLEL_JOBS: 4
Recommended BATCH_SIZE: 192
Recommended NUM_WORKERS: 1
Reason: Estimated usable GPU memory: 33.0 GB after reserve; estimated memory per job: 8.0 GB; CPU limit suggests up to 8 jobs.

Suggested config:
MAX_PARALLEL_JOBS = 4
BATCH_SIZE = 192
NUM_WORKERS = 1
USE_AMP = True
GPU_IDS = [0]

A100 40GB-like GPU detected: start with 2–3 jobs, maybe 4 

In [2]:
import os
import platform
import subprocess
import multiprocessing as mp
from pathlib import Path

import torch


def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT)
        return out.strip()
    except Exception as e:
        return f"ERROR: {e}"


def get_ram_gb():
    try:
        import psutil
        mem = psutil.virtual_memory()
        return mem.total / 1024**3, mem.available / 1024**3
    except Exception:
        return None, None


def recommend_jobs(
    gpu_mem_total_gb,
    gpu_mem_free_gb,
    cpu_count,
    batch_size=256,
    model_size="mixed",
):
    """
    Rough safe heuristic for your Transformer grid.

    model_size:
        small  — mostly T_small_mean
        base   — mostly T_base/T_base_cls
        mixed  — full grid including T_wide_mean/T_deep_mean
        wide   — mostly T_wide_mean
    """

    if gpu_mem_total_gb is None:
        return {
            "recommended_jobs": 1,
            "reason": "CUDA/GPU not detected. Use CPU mode.",
            "batch_size": min(batch_size, 64),
            "num_workers": 0,
        }

    # Conservative per-process memory estimate for your models with AMP.
    # These are intentionally cautious.
    if model_size == "small":
        mem_per_job = 4.0 if batch_size <= 256 else 6.0
    elif model_size == "base":
        mem_per_job = 6.0 if batch_size <= 256 else 8.0
    elif model_size == "wide":
        mem_per_job = 9.0 if batch_size <= 256 else 12.0
    else:
        mem_per_job = 8.0 if batch_size <= 256 else 10.0

    # Leave safety reserve for CUDA context, dataloaders, fragmentation.
    reserve_gb = max(6.0, 0.15 * gpu_mem_total_gb)
    usable_gb = max(0.0, gpu_mem_free_gb - reserve_gb)

    by_mem = int(usable_gb // mem_per_job)
    by_cpu = max(1, cpu_count // 4)

    recommended = max(1, min(by_mem, by_cpu))

    # Clamp to sensible range for one A100.
    if gpu_mem_total_gb >= 70:
        recommended = min(recommended, 6)
    elif gpu_mem_total_gb >= 35:
        recommended = min(recommended, 4)
    else:
        recommended = min(recommended, 2)

    if recommended <= 1:
        safe_batch = min(batch_size, 512)
        num_workers = 2
    elif recommended <= 2:
        safe_batch = min(batch_size, 256)
        num_workers = 2
    elif recommended <= 4:
        safe_batch = min(batch_size, 192)
        num_workers = 1
    else:
        safe_batch = min(batch_size, 128)
        num_workers = 1

    reason = (
        f"Estimated usable GPU memory: {usable_gb:.1f} GB after reserve; "
        f"estimated memory per job: {mem_per_job:.1f} GB; "
        f"CPU limit suggests up to {by_cpu} jobs."
    )

    return {
        "recommended_jobs": recommended,
        "reason": reason,
        "batch_size": safe_batch,
        "num_workers": num_workers,
        "usable_gpu_mem_gb": usable_gb,
        "estimated_mem_per_job_gb": mem_per_job,
        "cpu_limited_jobs": by_cpu,
        "memory_limited_jobs": by_mem,
    }


def inspect_system(batch_size=256, model_size="mixed"):
    print("=" * 90)
    print("SYSTEM")
    print("=" * 90)
    print("Platform:", platform.platform())
    print("Python:", platform.python_version())
    print("Working dir:", Path.cwd())
    print("CPU cores:", os.cpu_count())
    print("CPU count via multiprocessing:", mp.cpu_count())

    ram_total, ram_available = get_ram_gb()
    if ram_total is not None:
        print(f"RAM total: {ram_total:.2f} GB")
        print(f"RAM available: {ram_available:.2f} GB")
    else:
        print("RAM: psutil not available")

    print("\n" + "=" * 90)
    print("PYTORCH / CUDA")
    print("=" * 90)
    print("torch version:", torch.__version__)
    print("torch CUDA version:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())

    gpu_mem_total_gb = None
    gpu_mem_free_gb = None

    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("CUDA device count:", n)

        for i in range(n):
            props = torch.cuda.get_device_properties(i)
            total_gb = props.total_memory / 1024**3

            torch.cuda.set_device(i)
            free_bytes, total_bytes = torch.cuda.mem_get_info(i)
            free_gb = free_bytes / 1024**3
            total_runtime_gb = total_bytes / 1024**3

            print("\nGPU", i)
            print("Name:", props.name)
            print(f"Total memory by properties: {total_gb:.2f} GB")
            print(f"Free memory now: {free_gb:.2f} GB")
            print(f"Total memory runtime: {total_runtime_gb:.2f} GB")
            print("Compute capability:", f"{props.major}.{props.minor}")
            print("Multiprocessors:", props.multi_processor_count)

            if i == 0:
                gpu_mem_total_gb = total_runtime_gb
                gpu_mem_free_gb = free_gb
    else:
        print("No CUDA GPU visible to PyTorch.")

    print("\n" + "=" * 90)
    print("NVIDIA-SMI")
    print("=" * 90)
    print(run_cmd("nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu --format=csv,noheader,nounits"))

    print("\n" + "=" * 90)
    print("RECOMMENDATION")
    print("=" * 90)

    rec = recommend_jobs(
        gpu_mem_total_gb=gpu_mem_total_gb,
        gpu_mem_free_gb=gpu_mem_free_gb,
        cpu_count=os.cpu_count() or 1,
        batch_size=batch_size,
        model_size=model_size,
    )

    print("Model size assumption:", model_size)
    print("Input batch_size:", batch_size)
    print("Recommended MAX_PARALLEL_JOBS:", rec["recommended_jobs"])
    print("Recommended BATCH_SIZE:", rec["batch_size"])
    print("Recommended NUM_WORKERS:", rec["num_workers"])
    print("Reason:", rec["reason"])

    print("\nSuggested config:")
    print(f"MAX_PARALLEL_JOBS = {rec['recommended_jobs']}")
    print(f"BATCH_SIZE = {rec['batch_size']}")
    print(f"NUM_WORKERS = {rec['num_workers']}")
    print("USE_AMP = True")
    print("GPU_IDS = [0]")

    if torch.cuda.is_available() and gpu_mem_total_gb is not None:
        if gpu_mem_total_gb >= 70:
            print("\nA100 80GB-like GPU detected: you can test 4–6 jobs.")
        elif gpu_mem_total_gb >= 35:
            print("\nA100 40GB-like GPU detected: start with 2–3 jobs, maybe 4 if memory/utilization allow.")
        else:
            print("\nSmaller GPU detected: use 1–2 jobs.")

    return rec


# ============================================================
# RUN
# ============================================================

rec = inspect_system(
    batch_size=256,
    model_size="mixed",  # small / base / mixed / wide
)

SYSTEM
Platform: Linux-6.8.0-110-generic-x86_64-with-glibc2.39
Python: 3.12.3
Working dir: /home/user/fractional_unlearning
CPU cores: 32
CPU count via multiprocessing: 32
RAM total: 30.79 GB
RAM available: 23.78 GB

PYTORCH / CUDA
torch version: 2.6.0+cu124
torch CUDA version: 12.4
CUDA available: True
CUDA device count: 1

GPU 0
Name: NVIDIA A100-PCIE-40GB
Total memory by properties: 39.38 GB
Free memory now: 38.95 GB
Total memory runtime: 39.38 GB
Compute capability: 8.0
Multiprocessors: 108

NVIDIA-SMI
NVIDIA A100-PCIE-40GB, 40960, 437, 39890, 4

RECOMMENDATION
Model size assumption: mixed
Input batch_size: 256
Recommended MAX_PARALLEL_JOBS: 4
Recommended BATCH_SIZE: 192
Recommended NUM_WORKERS: 1
Reason: Estimated usable GPU memory: 33.0 GB after reserve; estimated memory per job: 8.0 GB; CPU limit suggests up to 8 jobs.

Suggested config:
MAX_PARALLEL_JOBS = 4
BATCH_SIZE = 192
NUM_WORKERS = 1
USE_AMP = True
GPU_IDS = [0]

A100 40GB-like GPU detected: start with 2–3 jobs, maybe 4 

In [2]:
from __future__ import annotations

import os
import sys
import json
import time
import shutil
import platform
import subprocess
from pathlib import Path


REPORT_PATH = Path.cwd() / "gpu_diagnostics_report.txt"


# ============================================================
# LOGGER
# ============================================================

LOG_LINES = []


def log(x=""):
    text = str(x)
    print(text)
    LOG_LINES.append(text)


def section(title):
    log("\n" + "=" * 100)
    log(title)
    log("=" * 100)


def run_cmd(cmd, timeout=20):
    log(f"\n$ {cmd}")
    try:
        out = subprocess.check_output(
            cmd,
            shell=True,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=timeout,
        )
        out = out.strip()
        log(out if out else "<empty output>")
        return out
    except subprocess.CalledProcessError as e:
        out = e.output.strip() if e.output else ""
        log(f"COMMAND FAILED, returncode={e.returncode}")
        log(out if out else "<empty output>")
        return None
    except subprocess.TimeoutExpired:
        log("COMMAND TIMEOUT")
        return None
    except Exception as e:
        log(f"COMMAND ERROR: {repr(e)}")
        return None


def save_report():
    REPORT_PATH.write_text("\n".join(LOG_LINES), encoding="utf-8")
    log(f"\nSaved report to: {REPORT_PATH}")


# ============================================================
# BASIC SYSTEM INFO
# ============================================================

section("1. BASIC SYSTEM INFO")

log("Current working directory:")
log(Path.cwd())

log("\nPlatform:")
log(platform.platform())

log("\nPython:")
log(sys.version)

log("\nPython executable:")
log(sys.executable)

log("\nPrefix:")
log(f"sys.prefix = {sys.prefix}")
log(f"sys.base_prefix = {sys.base_prefix}")

log("\nEnvironment:")
log(f"CONDA_PREFIX = {os.environ.get('CONDA_PREFIX')}")
log(f"VIRTUAL_ENV = {os.environ.get('VIRTUAL_ENV')}")
log(f"PATH = {os.environ.get('PATH')}")

log("\nUser:")
run_cmd("whoami")
run_cmd("hostname")
run_cmd("pwd")

log("\nCPU:")
run_cmd("nproc")
run_cmd("lscpu | head -80")

log("\nRAM:")
run_cmd("free -h")


# ============================================================
# IMPORTANT ENV VARS
# ============================================================

section("2. CUDA / GPU ENVIRONMENT VARIABLES")

env_keys = [
    "CUDA_VISIBLE_DEVICES",
    "CUDA_HOME",
    "CUDA_PATH",
    "LD_LIBRARY_PATH",
    "PATH",
    "NVIDIA_VISIBLE_DEVICES",
    "NVIDIA_DRIVER_CAPABILITIES",
    "CONDA_PREFIX",
    "VIRTUAL_ENV",
]

for k in env_keys:
    log(f"{k} = {os.environ.get(k)}")


# ============================================================
# SYSTEM GPU VISIBILITY
# ============================================================

section("3. SYSTEM-LEVEL GPU VISIBILITY")

log("Check if nvidia-smi exists:")
log(f"which nvidia-smi = {shutil.which('nvidia-smi')}")

run_cmd("nvidia-smi", timeout=30)

run_cmd(
    "nvidia-smi --query-gpu=index,name,driver_version,cuda_version,memory.total,memory.used,memory.free,utilization.gpu --format=csv",
    timeout=30,
)

log("\nPCI devices:")
run_cmd("lspci | grep -i nvidia", timeout=20)

log("\n/dev/nvidia*:")
run_cmd("ls -lah /dev/nvidia* 2>/dev/null || true")

log("\nNVIDIA proc driver version:")
run_cmd("cat /proc/driver/nvidia/version 2>/dev/null || true")

log("\nLoaded NVIDIA kernel modules:")
run_cmd("lsmod | grep -i nvidia || true")

log("\nCUDA compiler:")
run_cmd("which nvcc || true")
run_cmd("nvcc --version 2>/dev/null || true")


# ============================================================
# DOCKER / CONTAINER CHECKS
# ============================================================

section("4. DOCKER / CONTAINER CHECKS")

run_cmd("cat /proc/1/cgroup | head -50")
run_cmd("test -f /.dockerenv && echo 'INSIDE_DOCKER=YES' || echo 'INSIDE_DOCKER=NO'")
run_cmd("env | grep -E 'NVIDIA|CUDA|CONTAINER|DOCKER' || true")


# ============================================================
# PYTORCH CHECKS
# ============================================================

section("5. PYTORCH CHECKS")

try:
    import torch

    log(f"torch.__version__ = {torch.__version__}")
    log(f"torch.version.cuda = {torch.version.cuda}")
    log(f"torch.backends.cuda.is_built() = {torch.backends.cuda.is_built()}")
    log(f"torch.cuda.is_available() = {torch.cuda.is_available()}")
    log(f"torch.cuda.device_count() = {torch.cuda.device_count()}")

    try:
        log(f"torch.cuda.current_device() = {torch.cuda.current_device()}")
    except Exception as e:
        log(f"torch.cuda.current_device() ERROR: {repr(e)}")

    try:
        log(f"torch.cuda.get_arch_list() = {torch.cuda.get_arch_list()}")
    except Exception as e:
        log(f"torch.cuda.get_arch_list() ERROR: {repr(e)}")

    log("\nPyTorch config:")
    try:
        log(torch.__config__.show())
    except Exception as e:
        log(f"torch.__config__.show() ERROR: {repr(e)}")

    if torch.cuda.device_count() > 0:
        for i in range(torch.cuda.device_count()):
            section(f"5.{i}. PYTORCH GPU DEVICE {i}")

            try:
                props = torch.cuda.get_device_properties(i)
                log(f"name = {props.name}")
                log(f"total_memory_GB = {props.total_memory / 1024**3:.2f}")
                log(f"major.minor = {props.major}.{props.minor}")
                log(f"multi_processor_count = {props.multi_processor_count}")
            except Exception as e:
                log(f"get_device_properties ERROR: {repr(e)}")

            try:
                free_bytes, total_bytes = torch.cuda.mem_get_info(i)
                log(f"mem_get_info free_GB = {free_bytes / 1024**3:.2f}")
                log(f"mem_get_info total_GB = {total_bytes / 1024**3:.2f}")
            except Exception as e:
                log(f"mem_get_info ERROR: {repr(e)}")

except Exception as e:
    log(f"IMPORT TORCH ERROR: {repr(e)}")
    torch = None


# ============================================================
# PYTORCH GPU FUNCTIONAL TESTS
# ============================================================

section("6. PYTORCH GPU FUNCTIONAL TESTS")

if "torch" in globals() and torch is not None:
    if torch.cuda.is_available():
        try:
            device = torch.device("cuda:0")
            torch.cuda.set_device(0)

            log("CUDA device selected: cuda:0")
            log(f"Device name: {torch.cuda.get_device_name(0)}")

            log("\nTest 1: allocate small tensor on GPU")
            x = torch.randn(1024, 1024, device=device)
            log(f"x.device = {x.device}")
            log(f"x.shape = {tuple(x.shape)}")
            log(f"allocated_GB = {torch.cuda.memory_allocated(0) / 1024**3:.4f}")
            del x
            torch.cuda.empty_cache()

            log("\nTest 2: matrix multiplication benchmark")
            size = 4096
            a = torch.randn(size, size, device=device)
            b = torch.randn(size, size, device=device)

            torch.cuda.synchronize()
            t0 = time.time()
            c = a @ b
            torch.cuda.synchronize()
            t1 = time.time()

            log(f"matmul size = {size}x{size}")
            log(f"time_sec = {t1 - t0:.4f}")
            log(f"c.mean = {c.mean().item():.6f}")
            log(f"allocated_GB = {torch.cuda.memory_allocated(0) / 1024**3:.4f}")

            del a, b, c
            torch.cuda.empty_cache()

            log("\nTest 3: AMP autocast")
            a = torch.randn(2048, 2048, device=device)
            b = torch.randn(2048, 2048, device=device)

            with torch.amp.autocast(device_type="cuda", enabled=True):
                c = a @ b

            torch.cuda.synchronize()
            log(f"AMP c.dtype = {c.dtype}")
            log(f"AMP OK")

            del a, b, c
            torch.cuda.empty_cache()

            log("\nTest 4: tiny neural net training step")
            model = torch.nn.Sequential(
                torch.nn.Linear(512, 1024),
                torch.nn.GELU(),
                torch.nn.Linear(1024, 10),
            ).to(device)

            opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
            loss_fn = torch.nn.CrossEntropyLoss()
            scaler = torch.amp.GradScaler("cuda", enabled=True)

            inp = torch.randn(512, 512, device=device)
            target = torch.randint(0, 10, (512,), device=device)

            torch.cuda.synchronize()
            t0 = time.time()

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=True):
                out = model(inp)
                loss = loss_fn(out, target)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            torch.cuda.synchronize()
            t1 = time.time()

            log(f"tiny training loss = {loss.item():.6f}")
            log(f"tiny training step time_sec = {t1 - t0:.4f}")
            log("Tiny training OK")

            del model, opt, inp, target, out, loss
            torch.cuda.empty_cache()

        except Exception as e:
            log(f"GPU FUNCTIONAL TEST ERROR: {repr(e)}")
            log("Traceback:")
            import traceback
            log(traceback.format_exc())

    else:
        log("torch.cuda.is_available() is False, skipping GPU tensor tests.")
else:
    log("torch import failed, skipping PyTorch GPU tests.")


# ============================================================
# JUPYTER / KERNEL CHECKS
# ============================================================

section("7. JUPYTER / KERNEL CHECKS")

try:
    import ipykernel
    log(f"ipykernel version = {ipykernel.__version__}")
except Exception as e:
    log(f"ipykernel import error: {repr(e)}")

try:
    import notebook
    log(f"notebook version = {notebook.__version__}")
except Exception as e:
    log(f"notebook import error: {repr(e)}")

try:
    import jupyter_core
    log(f"jupyter_core version = {jupyter_core.__version__}")
except Exception as e:
    log(f"jupyter_core import error: {repr(e)}")

run_cmd("jupyter kernelspec list 2>/dev/null || true")
run_cmd("python -m ipykernel --version 2>/dev/null || true")


# ============================================================
# PYTHON / PIP PACKAGE CHECKS
# ============================================================

section("8. PYTHON PACKAGE CHECKS")

run_cmd(f"{sys.executable} -m pip show torch torchvision torchaudio 2>/dev/null || true", timeout=30)
run_cmd(f"{sys.executable} -m pip list | grep -E 'torch|cuda|nvidia|triton' || true", timeout=30)

log("\nTry import common NVIDIA Python packages:")
for pkg in [
    "torch",
    "triton",
    "nvidia",
]:
    try:
        __import__(pkg)
        log(f"{pkg}: OK")
    except Exception as e:
        log(f"{pkg}: ERROR {repr(e)}")


# ============================================================
# EXTERNAL PYTHON SUBPROCESS CHECK
# ============================================================

section("9. SUBPROCESS PYTHON CUDA CHECK")

cmd = (
    f"{sys.executable} -c "
    "\"import torch,sys,os;"
    "print('exe=',sys.executable);"
    "print('torch=',torch.__version__);"
    "print('torch_cuda=',torch.version.cuda);"
    "print('cuda_available=',torch.cuda.is_available());"
    "print('device_count=',torch.cuda.device_count());"
    "print('CUDA_VISIBLE_DEVICES=',os.environ.get('CUDA_VISIBLE_DEVICES'));"
    "print('names=',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\""
)
run_cmd(cmd, timeout=30)


# ============================================================
# DIAGNOSIS
# ============================================================

section("10. DIAGNOSIS / WHAT IT MEANS")

diagnosis = []

# Collect signals
nvidia_smi_exists = shutil.which("nvidia-smi") is not None
cuda_available = False
torch_cuda_version = None
torch_cuda_built = None
torch_device_count = 0

if "torch" in globals() and torch is not None:
    try:
        cuda_available = torch.cuda.is_available()
        torch_cuda_version = torch.version.cuda
        torch_cuda_built = torch.backends.cuda.is_built()
        torch_device_count = torch.cuda.device_count()
    except Exception:
        pass

log(f"nvidia-smi exists: {nvidia_smi_exists}")
log(f"torch CUDA built: {torch_cuda_built}")
log(f"torch.version.cuda: {torch_cuda_version}")
log(f"torch.cuda.is_available: {cuda_available}")
log(f"torch.cuda.device_count: {torch_device_count}")

if not nvidia_smi_exists:
    diagnosis.append(
        "nvidia-smi is not found. Either NVIDIA driver is not installed, "
        "or this environment/container does not expose NVIDIA tools."
    )

if torch_cuda_version is None:
    diagnosis.append(
        "PyTorch appears to be CPU-only or CUDA support is not built into this torch installation "
        "(torch.version.cuda is None). Install CUDA-enabled PyTorch."
    )

if nvidia_smi_exists and torch_cuda_version is not None and not cuda_available:
    diagnosis.append(
        "nvidia-smi exists and PyTorch has CUDA build, but torch.cuda.is_available() is False. "
        "Likely causes: driver too old for this PyTorch CUDA build, GPU not passed into container, "
        "CUDA_VISIBLE_DEVICES hides the GPU, or Jupyter kernel uses a broken environment."
    )

if cuda_available and torch_device_count > 0:
    diagnosis.append(
        "PyTorch sees CUDA GPU(s). Training code should use DEVICE='cuda'. "
        "If your training still uses CPU, check that the notebook/script is running in this same Python kernel."
    )

if not diagnosis:
    diagnosis.append("No clear diagnosis. Check the raw logs above.")

for i, d in enumerate(diagnosis, 1):
    log(f"{i}. {d}")


# ============================================================
# RECOMMENDED ACTIONS
# ============================================================

section("11. RECOMMENDED ACTIONS")

log("Case A: nvidia-smi fails or no /dev/nvidia0")
log(
    "  -> You are probably on a CPU node/container, or GPU is not passed through. "
    "Use the correct GPU node, request GPU in SLURM, or start Docker with --gpus all."
)

log("\nCase B: nvidia-smi works, but torch.version.cuda is None")
log("  -> Installed PyTorch is CPU-only. Reinstall CUDA PyTorch:")
log("     pip uninstall -y torch torchvision torchaudio")
log("     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
log("  If driver is old, try cu118:")
log("     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

log("\nCase C: nvidia-smi works, torch.version.cuda is not None, but torch.cuda.is_available() is False")
log("  -> Usually driver/PyTorch CUDA mismatch or container issue.")
log("  Check driver version in nvidia-smi.")
log("  If driver is old, install PyTorch with older CUDA, e.g. cu118.")
log("  If inside Docker, restart container with:")
log("     docker run --gpus all ...")
log("  If SLURM/cluster, request GPU:")
log("     srun --gres=gpu:a100:1 --pty bash")

log("\nCase D: Terminal sees GPU, Jupyter does not")
log("  -> Jupyter kernel is not the same Python environment.")
log("  Compare:")
log("     which python")
log("     python -c \"import torch; print(torch.cuda.is_available())\"")
log("  with this notebook:")
log("     import sys; print(sys.executable)")
log("  Then install/register correct kernel:")
log("     python -m pip install ipykernel")
log("     python -m ipykernel install --user --name gpu-env --display-name 'Python GPU env'")

log("\nCase E: CUDA works")
log("  -> In training code set:")
log("     DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')")
log("     USE_AMP = True")
log("     BATCH_SIZE = 256 or 512")
log("     MAX_PARALLEL_JOBS = 2..5 depending on memory/utilization")


# ============================================================
# SAVE REPORT
# ============================================================

section("12. SAVE REPORT")
save_report()


1. BASIC SYSTEM INFO
Current working directory:
/home/user/fractional_unlearning

Platform:
Linux-6.8.0-110-generic-x86_64-with-glibc2.39

Python:
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

Python executable:
/usr/bin/python3

Prefix:
sys.prefix = /usr
sys.base_prefix = /usr

Environment:
CONDA_PREFIX = None
VIRTUAL_ENV = None
PATH = /home/tahiti/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/snap/bin

User:

$ whoami
tahiti

$ hostname
tahiti-Standard-PC-i440FX-PIIX-1996

$ pwd
/home/user/fractional_unlearning

CPU:

$ nproc
32

$ lscpu | head -80
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           40 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  32
On-line CPU(s) list:                     0-31
Vendor ID:                               Genuine

<empty output>

$ python -m ipykernel --version 2>/dev/null || true
<empty output>

8. PYTHON PACKAGE CHECKS

$ /usr/bin/python3 -m pip show torch torchvision torchaudio 2>/dev/null || true
Name: torch
Version: 2.11.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: 
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /home/tahiti/.local/lib/python3.12/site-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by:

$ /usr/bin/python3 -m pip list | grep -E 'torch|cuda|nvidia|triton' || true
cuda-bindings               13.2.0
cuda-pathfinder             1.5.4
cuda-toolkit                13.0.2
nvidia-cublas               13.1.0.3
nvidia-cuda-cupti           13.0.85
nvidia-cuda-nvrtc           13.0.88
nvidia-cuda-runtime         13.0.96
nvidi

In [1]:
import sys, torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

/home/user/fractional_unlearning/.venv_a100/bin/python
2.6.0+cu124
12.4
True
NVIDIA A100-PCIE-40GB
